# Multimodal Emotion Recognition — CREMA-D
Thin wrapper — all logic lives in the `emotion_recognition/` package.
Upload the folder to Google Drive and set `PACKAGE_DIR` below.

In [ ]:
! git clone https://github.com/Paresh140/Multimodal_Voice

In [ ]:
! ls


In [ ]:
! cd /kaggle/working/Multimodal_Voice

In [ ]:
! ls

In [ ]:
%cd /content/Multimodal_Voice
!git pull origin main

In [ ]:
! pwd

In [ ]:
! ls

In [ ]:
# ── 2. Install deps ─────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchaudio', 'transformers==4.49',
    'soundfile', 'scikit-learn', 'pandas', 'numpy',
    'matplotlib', 'seaborn', 'tqdm'])

In [ ]:
# ── 3. Point Python at the package ─────────────────────────────────────────
import sys

# *** EDIT THESE TWO PATHS ***
PACKAGE_DIR = '/kaggle/working/Multimodal_Voice/emotion_recognition'
DATA_ROOT   = '/kaggle/input/datasets/ejlok1/cremad/AudioWAV'
OUTPUT_DIR  = '/kaggle/working/'

sys.path.insert(0, PACKAGE_DIR)

### 4. Authenticate with Hugging Face
Make sure you have added `HF_TOKEN` to your Colab Secrets (the 🔑 icon on the left sidebar) and granted notebook access.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
except userdata.SecretNotFoundError:
    print("Warning: HF_TOKEN secret not found. Please add it in the Secrets tab (🔑) on the left.")

In [ ]:
# ── 5. Run ──────────────────────────────────────────────────────────────────
from config import Config
from main import main

cfg = Config(
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    use_text=True,
    batch_size=8,
    grad_accum_steps=8,  # was 4
    num_epochs=50,              # was 20 — give it room
    use_fp16=True,
    freeze_transformer_layers=8,  # int, not 10. — confirmed this is what worked   
)


results = main(cfg)
print(f"\nFinal test accuracy : {results['acc']:.4f}")
print(f"Final test macro-F1 : {results['macro_f1']:.4f}")

In [ ]:
! ls

In [ ]:
!pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
! pip install --upgrade bitsandbytes

In [ ]:
!pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu124

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Supported Architectures: {torch.cuda.get_arch_list()}")

In [ ]:
# Force-reinstall a stable version that includes sm_60 support
!pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
pip install transformers==4.49

In [ ]:
# ── 6. 5-fold Cross-Validation (speaker-grouped) ─────────────────────────────
# Runs StratifiedGroupKFold on train+val (speaker-grouped by actor_id) and prints
# mean ± std across folds. Test set from speaker_independent_split stays untouched.
import importlib

import cross_validate
importlib.reload(cross_validate)  # ensure latest version is loaded in the notebook kernel

# Reuse cfg from the training cell above
mean_acc, mean_f1 = cross_validate.cross_validate(cfg)
print(f"\n5-fold CV mean accuracy: {mean_acc:.4f}")
print(f"5-fold CV mean macro-F1: {mean_f1:.4f}")